## Solución Taller 03 - Optimización de Redes Metabólicas

In [2]:
# Librerías a utilizar en el taller
import numpy as np
from gurobipy import Model, GRB, quicksum

### Pregunta a)

Se solicita ver el flujo máximo de R3, sujeto a que R1 ni R2 pueden superar 10 unidades (absoluto). Se debe tener en cuenta que solo las reacciones R2 y R8 son reversibles.

In [20]:
# Vector objetivo: maximizar r3
f = np.zeros(10)
f[2] = 1  # r3

# Matriz de restricciones N
N = np.array([
    [1,0,0,0,-1,-1,-1,0,0,0],
    [0,1,0,0,1,0,0,-1,-1,0],
    [0,0,0,0,0,1,0,1,0,-1],
    [0,0,0,0,0,0,1,0,0,-1],
    [0,0,0,-1,0,0,0,0,0,1],
    [0,0,-1,0,0,0,0,0,1,1]
])
b = np.zeros(6)

# Límites

# Inferiores
lb    = np.zeros(10) # Irreversibles excepto r2 y r8
lb[1] = -10          # Indican que no puede superar 10
lb[7] = -1e6         # Reversible irrestricta (número muy grande)

# Superiores
ub    = 1e6*np.ones(10)  # Límites irrestrictos (número muy grande)
ub[0] = 10 # r1 no puede superar 10
ub[1] = 10 # r2 no puede superar 10

model_a = Model()

# Añadir variables
r = []
for i in range(10):
    r.append(model_a.addVar(lb=lb[i], ub=ub[i], name=f"r_{i+1}"))

# Añadir restricciones
for i in range(6):
    model_a.addConstr(quicksum(N[i,j]*r[j] for j in range(10)) == b[i])

# Función objetivo: maximizar r3
model_a.setObjective(r[2], GRB.MAXIMIZE)

# sin output (opcional)
model_a.setParam('OutputFlag', 0)

# Resolver
model_a.optimize()

fmax_r3_a = model_a.ObjVal
print(f"Flujo máximo por r3 es {fmax_r3_a}")

Flujo máximo por r3 es 20.0


### Pregunta b)

En esta pregunta, se debe resolver un problema MILP para identificar un mínimo de reacciones que permita generar un flujo por R3 = 20.0. Para ello, se añade un límite inferior a R3 igual a 20.0 y se crean variables binarias que "apagan" o "prenden" variables. Esto se realiza multiplicando los límites inferiores y superiores por variables binarias $y_i$.

In [47]:
model_b = Model()

# Variables continuas r
r     = []
lb[2] = fmax_r3_a  # Se debe producir 20.0
ub[2] = fmax_r3_a  # Se debe producir 20.0

for i in range(10):
    r.append(model_b.addVar(lb=lb[i], ub=ub[i], name=f"r_{i+1}"))

# Variables binarias y
y = []
for i in range(10):
    y.append(model_b.addVar(vtype=GRB.BINARY, name=f"y_{i+1}"))

# Función objetivo: minimizar suma de y_i
model_b.setObjective(quicksum(y[i] for i in range(10)), GRB.MINIMIZE)

# Restricción N*r = 0
for i in range(6):
    model_b.addConstr(quicksum(N[i,j]*r[j] for j in range(10)) == b[i])

# Restricciones para "apagar" o "prender" los flujos
for i in range(10):
    model_b.addConstr(r[i] <= ub[i]*y[i])
    model_b.addConstr(r[i] >= lb[i]*y[i])

# sin output (opcional)
model_b.setParam('OutputFlag', 0)

# Resolver
model_b.optimize()

# Extraer solución
xmin_yi_b = [y[i].X for i in range(10)]
fmin_yi_b = model_b.ObjVal

print(f"Solución óptima yi: {[int(val) for val in xmin_yi_b]}")
print(f"Cantidad mínima de flujos: {int(fmin_yi_b)}")


Solución óptima yi: [1, 1, 1, 0, 1, 0, 0, 0, 1, 0]
Cantidad mínima de flujos: 5


### Pregunta c)
Para enumerar todas las combinaciones de soluciones que permiten alcanzar el flujo R3 = 20.0, se deben añadir restricciones del tipo *integer cut*:

Si ya encontraste una solución óptima $y^*$, agregas un corte del tipo:

$$
\sum_{i=1}^{10} y^*_i \cdot y_i \leq \sum_{i=1}^{10} y^*_i - 1
$$

Esto obliga al solver a buscar una combinación distinta de variables binarias activas, pero manteniendo el mismo valor óptimo.

Se realiza esto de forma iterativa, añadiendo una restricción cada vez que se obtiene una solución óptima, hasta que se vuelva infactible.


In [ ]:
# Guardamos la primera solución
xsol = np.array([var.X for var in r + y]).reshape(-1,1)

while True:
    # Obtener la última solución binaria
    ysol = np.array([var.X for var in y])
    yopt = np.floor(ysol).astype(int)

    # Cardinalidad
    Mopt = int(sum(yopt))

    # Nueva restricción tipo cut: sum(yopt * y) <= Mopt - 1
    model_b.addConstr(quicksum(yopt[i]*y[i] for i in range(10)) <= Mopt - 1)

    # Resolver de nuevo
    model_b.optimize()

    if model_b.Status == GRB.OPTIMAL:
        new_sol = np.array([var.X for var in r + y]).reshape(-1,1)
        xsol = np.hstack((xsol, new_sol))
    else:
        break

# Mostrar todas las soluciones binarias encontradas
print("Soluciones yi encontradas:") 
print(xsol[10:,:])

print("\nDistribuciones de flujos:")
print(xsol[:10,:])


Soluciones yi encontradas:
[[ 1.  1.]
 [ 1.  1.]
 [ 1.  1.]
 [-0.  0.]
 [ 1.  0.]
 [-0.  1.]
 [ 0.  0.]
 [-0.  1.]
 [ 1.  1.]
 [ 0.  0.]]

Distribuciones de flujos:
[[ 10.  10.]
 [ 10.  10.]
 [ 20.  20.]
 [  0.   0.]
 [ 10.   0.]
 [  0.  10.]
 [  0.   0.]
 [  0. -10.]
 [ 20.  20.]
 [  0.   0.]]
